# Width-Scaling SGD Experiments with Causal Attention, d16 Excluded

Same task, optimizer, and width sweep as `colab_width_scaling_sgd_exclude_d8_d16.ipynb`,
with `use_attention: True` selecting the transformer architecture instead of the plain
residual MLP.

The transformer treats the problem as next-position prediction. The sequence is the 16
input bits followed by the target parities in binary-tree order (all degree-2 parities,
then degree-4, then degree-8). The last input position predicts the first degree-2
parity, the next position predicts the second, and so on. Every position — input bits
and intermediate answers alike — owns a learnable embedding vector that the value at
that position scales. Each of the `L` blocks is causal self-attention with a residual
connection followed by the same MLP block the residual net uses (also residual, with
`use_post_activation_linear` optional). There is no layer normalization. The unembedding
is a single position-independent `N -> 1` map.

Training is teacher-forced on the true parities. Test-time evaluation is autoregressive:
only the input bits are given and each prediction is fed back into the next position.

## Setup

Mount Google Drive, clone the public repo, install it in editable mode, and define the run/plot directories.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [11]:
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/labofdoubt/feature-learning-parity-task.git"
REPO_DIR = Path("/content/feature-learning-parity-task")

DRIVE_ROOT = Path("/content/drive/MyDrive/ml_projects_new/parity_width_scaling_sgd_exclude_d16_attention_fixed_emb_1_block")

RUNS_DIR = DRIVE_ROOT / "runs"
PLOTS_DIR = DRIVE_ROOT / "plots"
ANALYSIS_DIR = DRIVE_ROOT / "analysis"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
!rm -rf "$REPO_DIR"
!git clone "$GITHUB_REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!pip install -e .

## Train Width Sweep

One config per width, identical to the residual-MLP sweep except for `use_attention: True`
and `num_heads`. Existing final checkpoints are skipped unless `FORCE_RETRAIN = True`.


In [7]:
from pathlib import Path

import yaml

from parity_net.config import load_config
from parity_net.train import train

WIDTHS = [2048]
FORCE_RETRAIN = False


def make_config(N: int, output_dir: Path) -> dict:
    return {
        "model": {
            "input_dim": 16,
            "relevant_dim": 8,
            "N": N,
            "L": 3,
            "activation": "half-tanh",
            "use_readout_barrier": False,
            "embedding_weight_variance": 1.0 / 32,
            "freeze_embedding": False,
            "hidden_weight_variance": 1.0 / N,
            "readout_weight_variance": 1.0 / N,
            "use_layerwise_readouts": False,
            "use_post_activation_linear": True,
            "bias": False,
            "use_attention": True,
            "num_heads": 1,
            "attention_logit_scale": "1/sqrt(d)",
            "autoregressive_feedback": "raw",
            "use_kv_cache": True,
        },
        "task": {
            "input_dim": 16,
            "relevant_dim": 8,
            # "exclude_targets": ["d4", "d8", "d16"],
            "exclude_targets": ["d16"]
        },
        "training": {
            "num_steps": 10_000,
            "test_samples": 100_000,
            "batch_size": 512,
            "seed": 0,
            "device": "cuda",
            "dtype": "float32",
            "log_every": 1_000,
            "checkpoint_every": 10_000,
            "output_dir": str(output_dir),
            "barrier_c": None,
            "barrier_lambda": 10.0,
            "optimizer": {
                "name": "sgd",
                "lr": 1e-3,
                "lr_embedding": None,
                "lr_hidden": None,
                "lr_readout": None,
                "weight_decay": 1e-3,
                "wd_embedding": None,
                "wd_hidden": None,
                "wd_readout": None,
                "momentum": 0.9,
                "betas": [0.9, 0.999],
            },
        },
    }

def make_config_mup(N: int, output_dir: Path) -> dict:
    config = make_config(N, output_dir)
    optimizer = config["training"]["optimizer"]
    base_lr = optimizer["lr"]
    base_wd = optimizer["weight_decay"]
    config["model"]["readout_weight_variance"] = 1 / N**2
    # muP keeps query-key logits Theta(1) as head_dim grows with width.
    config["model"]["attention_logit_scale"] = "1/d"

    optimizer["lr_embedding"] = base_lr * N / 256
    optimizer["lr_hidden"] = base_lr
    optimizer["lr_readout"] = base_lr * 256 / N

    optimizer["wd_embedding"] = base_wd * 256 / N
    optimizer["wd_hidden"] = base_wd
    optimizer["wd_readout"] = base_wd * N / 256
    return config


CONFIG_FACTORY = make_config_mup  # Change to make_config for the unscaled sweep.

In [8]:
# config_paths = {}
# for N in WIDTHS:
#     run_dir = RUNS_DIR / f"N_{N}"
#     run_dir.mkdir(parents=True, exist_ok=True)
#     config_path = run_dir / "config.yaml"
#     final_checkpoint = run_dir / "checkpoints" / "final.pt"

#     if final_checkpoint.exists() and not FORCE_RETRAIN:
#         config_paths[N] = config_path
#         print(
#             f"N={N}: final checkpoint exists, skipping training: {final_checkpoint}. "
#             "Set FORCE_RETRAIN=True or choose a new DRIVE_ROOT to train with changed config values."
#         )
#         continue

#     config = CONFIG_FACTORY(N, run_dir)
#     with config_path.open("w") as f:
#         yaml.safe_dump(config, f, sort_keys=False)
#     config_paths[N] = config_path

#     print(f"N={N}: training with {config_path}")
#     final_path = train(load_config(config_path))
#     print(f"N={N}: saved final checkpoint to {final_path}")

In [ ]:
# from google.colab import runtime
# runtime.unassign()

## Train/Test Curves

Read `metrics.csv` from each run, save one plot per width, and save combined train/test
plots across widths. `test_mse` is autoregressive.

In [12]:
RUNS_DIR = DRIVE_ROOT / "runs"
PLOTS_DIR = DRIVE_ROOT / "plots"
ANALYSIS_DIR = DRIVE_ROOT / "analysis"

WIDTHS = [1024]

In [13]:
import matplotlib.pyplot as plt
import pandas as pd

USE_LOG_MSE_AXIS = True
USE_LOG_STEP_AXIS = True
TEST_MSE_COLUMNS = ["test_mse", "test_mse_d2", "test_mse_d4", "test_mse_d8", "test_mse_d16"]


def axis_has_positive_data(ax, axis):
    for line in ax.lines:
        values = line.get_xdata() if axis == "x" else line.get_ydata()
        if len(values) and pd.Series(values).dropna().gt(0).any():
            return True
    return False


def maybe_log_y(ax):
    if USE_LOG_MSE_AXIS and axis_has_positive_data(ax, "y"):
        ax.set_yscale("log")


def maybe_log_x(ax):
    if USE_LOG_STEP_AXIS and axis_has_positive_data(ax, "x"):
        ax.set_xscale("log")


metrics_by_width = {}
for N in WIDTHS:
    metrics_path = RUNS_DIR / f"N_{N}" / "metrics.csv"
    if not metrics_path.exists():
        print(f"N={N}: missing {metrics_path}")
        continue
    df = pd.read_csv(metrics_path)
    metrics_by_width[N] = df

    fig, ax = plt.subplots(figsize=(8, 5))
    for column in TEST_MSE_COLUMNS:
        if column in df.columns:
            ax.plot(df["step"], df[column], label=column)
    ax.set_xlabel("Step")
    ax.set_ylabel("MSE")
    ax.set_title(f"Test MSE by degree, N={N}")
    maybe_log_x(ax)
    maybe_log_y(ax)
    ax.grid(True, alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"test_mse_by_degree_N_{N}.png", dpi=150)
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df["step"], df["train_mse"], label="train_mse")
    ax.plot(df["step"], df["test_mse"], label="test_mse")
    ax.set_xlabel("Step")
    ax.set_ylabel("MSE")
    ax.set_title(f"Train/Test MSE, N={N}")
    maybe_log_x(ax)
    maybe_log_y(ax)
    ax.grid(True, alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"train_test_mse_N_{N}.png", dpi=150)
    plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for N, df in metrics_by_width.items():
    ax.plot(df["step"], df["test_mse"], label=f"N={N}")
ax.set_xlabel("Step")
ax.set_ylabel("Test MSE")
ax.set_title("Test MSE vs Step")
maybe_log_x(ax)
maybe_log_y(ax)
ax.grid(True, alpha=0.3, which="both")
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / "test_mse_by_width.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for N, df in metrics_by_width.items():
    ax.plot(df["step"], df["train_mse"], label=f"N={N}")
ax.set_xlabel("Step")
ax.set_ylabel("Train MSE")
ax.set_title("Train MSE vs Step")
maybe_log_x(ax)
maybe_log_y(ax)
ax.grid(True, alpha=0.3, which="both")
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / "train_mse_by_width.png", dpi=150)
plt.show()

## Attention Patterns

Attention weights for the sequence the model actually sees at test time: the input
bits followed by its own fed-back predictions. The sequence is produced
autoregressively first, then replayed in a single full-sequence pass, and each
layer's softmax is recomputed from the residual stream entering that layer. Set
`ATTN_FEED_MODEL_PREDICTIONS = False` to teacher-force the true parities instead.

Keys are labelled by the value a position carries (`x1`..`x16`, then the fed-back
parities); queries by the target that position predicts, so the last input bit
`x16` appears as `->d2_0`. The line separates input positions from answer positions,
and cells above the diagonal are masked by causality rather than merely small.

In [21]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from parity_net.analysis import resolve_test_data_path
from parity_net.checkpoint import load_checkpoint
from parity_net.data import load_dataset
from parity_net.train import resolve_device, resolve_dtype

ATTN_N = 1024
ATTN_CHECKPOINT_STEP = 90000  # Use "final" or an integer step, e.g. 10000.
ATTN_FEED_MODEL_PREDICTIONS = True  # True replays the test-time sequence (the model's own
                                    # fed-back predictions); False teacher-forces true parities.
ATTN_SAVE = True


def attn_checkpoint_name(checkpoint_step):
    if checkpoint_step == "final":
        return "final.pt", "final"
    if isinstance(checkpoint_step, int):
        return f"step_{checkpoint_step:08d}.pt", f"step_{checkpoint_step:08d}"
    raise ValueError('ATTN_CHECKPOINT_STEP must be "final" or an integer step')


attn_checkpoint_file, attn_checkpoint_label = attn_checkpoint_name(ATTN_CHECKPOINT_STEP)
attn_run_dir = RUNS_DIR / f"N_{ATTN_N}"
attn_checkpoint_path = attn_run_dir / "checkpoints" / attn_checkpoint_file
if not attn_checkpoint_path.exists():
    checkpoint_dir = attn_run_dir / "checkpoints"
    available = (
        sorted(p.name for p in checkpoint_dir.iterdir()) if checkpoint_dir.is_dir() else []
    )
    raise FileNotFoundError(
        f"Missing checkpoint: {attn_checkpoint_path}\n"
        f"Available in {checkpoint_dir}: {available or 'nothing'}\n"
        "Intermediate checkpoints are only written when step % checkpoint_every == 0, "
        "so lower training.checkpoint_every if you need a finer grid."
    )

load_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
attn_model, attn_payload, _ = load_checkpoint(attn_checkpoint_path, load_device)
attn_config = attn_payload["config"]
attn_training = attn_config["training"]
attn_model_config = attn_config["model"]
if not attn_model_config.get("use_attention", False):
    raise ValueError("This checkpoint was not trained with use_attention=True")

attn_device = resolve_device(attn_training["device"])
attn_dtype = resolve_dtype(attn_training["dtype"])
attn_model = attn_model.to(device=attn_device, dtype=attn_dtype).eval()
attn_batch_size = int(attn_training["batch_size"])

attn_test_data_path = resolve_test_data_path(
    attn_checkpoint_path, attn_training, attn_payload.get("test_data_path")
)
if attn_test_data_path is None:
    raise FileNotFoundError("Could not find saved test_data.pt for this checkpoint")
attn_test_data = load_dataset(attn_test_data_path, attn_device, attn_dtype)

attn_target_names = list(attn_model.target_names)
attn_input_dim = int(attn_model.config.input_dim)
attn_num_positions = int(attn_model.num_positions)
attn_num_heads = int(attn_model.config.num_heads)
attn_num_layers = len(attn_model.blocks)

# Keys are labelled by the value a position carries; queries by the target that
# position predicts. Position input_dim-1 (the last input bit) predicts the first
# target, so the two labellings are offset by one.
attn_key_labels = [f"x{i + 1}" for i in range(attn_input_dim)] + attn_target_names[
    : attn_num_positions - attn_input_dim
]
attn_query_labels = np.array(
    [f"x{i + 1}" for i in range(attn_input_dim - 1)] + list(attn_target_names)
)
assert len(attn_key_labels) == len(attn_query_labels) == attn_num_positions

attn_causal_mask = torch.tril(
    torch.ones(attn_num_positions, attn_num_positions, dtype=torch.bool, device=attn_device)
)


@torch.no_grad()
def attn_sequence_targets(x_batch, y_batch):
    """The answer-position values that fill the sequence."""
    if ATTN_FEED_MODEL_PREDICTIONS:
        return attn_model.feedback_value(attn_model.generate(x_batch))
    return y_batch


@torch.no_grad()
def attention_weights(x_batch, y_batch):
    """(layers, batch, heads, S, S) attention probabilities.

    Runs one full-sequence pass over the sequence the model sees at test time, then
    recomputes each layer's softmax from the residual stream entering that layer.
    activations[i] is the stream entering block i, which is exactly what that block's
    attention reads.
    """
    seq_targets = attn_sequence_targets(x_batch, y_batch)
    _, activations = attn_model(x_batch, targets=seq_targets, return_activations=True)
    per_layer = []
    for layer_idx, block in enumerate(attn_model.blocks):
        attention = block.attention
        h = activations[layer_idx]
        q = attention._split_heads(attention.q_proj(h))
        k = attention._split_heads(attention.k_proj(h))
        logits = (q @ k.transpose(-1, -2)) * attention.logit_scale
        per_layer.append(logits.masked_fill(~attn_causal_mask, float("-inf")).softmax(dim=-1))
    return torch.stack(per_layer)


@torch.no_grad()
def attn_verify_recomputation(num_samples=8):
    """Feed the recomputed weights back through V and O; the result must equal the
    block's own attention output, which proves these are the model's real weights."""
    x_batch = attn_test_data.x[:num_samples]
    y_batch = attn_test_data.y[:num_samples]
    seq_targets = attn_sequence_targets(x_batch, y_batch)
    _, activations = attn_model(x_batch, targets=seq_targets, return_activations=True)
    weights = attention_weights(x_batch, y_batch)
    worst = 0.0
    for layer_idx, block in enumerate(attn_model.blocks):
        attention = block.attention
        h = activations[layer_idx]
        v = attention._split_heads(attention.v_proj(h))
        rebuilt = attention.out_proj(
            (weights[layer_idx] @ v).transpose(1, 2).reshape(h.shape)
        )
        worst = max(worst, float((rebuilt - attention(h)).abs().max()))
    return worst


# Row length grows from 1 to S, so uniform attention alone spans a 22x range and a
# single colour scale cannot serve every row. "predictions" shows only the rows that
# emit a target - the ones labelled by parity name - where row lengths are similar and
# one scale works. "all" shows the full square including the pure-context rows.
ATTN_QUERY_ROWS = "predictions"
ATTN_VMAX = "auto"  # "auto" scales to the displayed rows, or pass a number.


def attn_row_slice():
    if ATTN_QUERY_ROWS == "all":
        return slice(0, attn_num_positions)
    if ATTN_QUERY_ROWS == "predictions":
        return slice(attn_input_dim - 1, attn_num_positions)
    raise ValueError('ATTN_QUERY_ROWS must be "predictions" or "all"')


def plot_attention_grid(weights, suptitle, filename):
    """weights: (layers, heads, S, S). Rows are heads, columns are blocks."""
    mask = attn_causal_mask.detach().cpu().numpy()
    rows = attn_row_slice()
    # Future positions are impossible, not zero, so they are left blank rather than
    # painted at the bottom of the ramp.
    shown = np.where(mask, weights, np.nan)[:, :, rows, :]
    row_labels = attn_query_labels[rows]
    vmax = float(ATTN_VMAX) if isinstance(ATTN_VMAX, (int, float)) else float(np.nanmax(shown))
    vmax = vmax or 1.0
    clipped = int(np.nansum(shown > vmax))
    cmap = plt.get_cmap("Blues").copy()
    cmap.set_bad("#F2F2F2")

    cell = 0.30
    width = cell * attn_num_positions + 1.6
    height = cell * len(row_labels) + 2.2
    fig, axes = plt.subplots(
        attn_num_heads,
        attn_num_layers,
        figsize=(width * attn_num_layers, height * attn_num_heads),
        squeeze=False,
        constrained_layout=True,
    )
    for layer_idx in range(attn_num_layers):
        for head_idx in range(attn_num_heads):
            ax = axes[head_idx][layer_idx]
            image = ax.imshow(
                shown[layer_idx, head_idx],
                cmap=cmap,
                vmin=0.0,
                vmax=vmax,
                interpolation="nearest",
            )
            head_suffix = f", head {head_idx + 1}" if attn_num_heads > 1 else ""
            ax.set_title(f"block {layer_idx + 1}{head_suffix}", fontsize=10)
            ax.set_xticks(range(attn_num_positions))
            ax.set_xticklabels(attn_key_labels, rotation=90, fontsize=7)
            ax.set_yticks(range(len(row_labels)))
            ax.set_yticklabels(row_labels, fontsize=7)
            ax.set_xlabel("key: value carried by position", fontsize=9)
            if layer_idx == 0:
                ax.set_ylabel("query: target predicted", fontsize=9)
            # Separate input-bit keys from fed-back answer keys.
            ax.axvline(attn_input_dim - 0.5, color="#555555", linewidth=0.9)
            if rows.start == 0:
                ax.axhline(attn_input_dim - 0.5, color="#555555", linewidth=0.9)
            ax.tick_params(length=0)
    label = "attention weight"
    if clipped:
        label += f" (>{vmax:.3g} clipped: {clipped} cells)"
    fig.colorbar(image, ax=axes, label=label, fraction=0.03, pad=0.01)
    fig.suptitle(suptitle, fontsize=12)
    if ATTN_SAVE:
        path = ANALYSIS_DIR / filename
        fig.savefig(path, dpi=170, bbox_inches="tight")
        print(f"Saved {path}")
    plt.show()


print(f"Checkpoint: {attn_checkpoint_path}")
print(f"Sequence: {attn_num_positions} positions = {attn_input_dim} input bits + "
      f"{attn_num_positions - attn_input_dim} fed-back answers; targets {attn_target_names}")
print(f"{attn_num_layers} blocks x {attn_num_heads} head(s); "
      f"sequence filled with {'model predictions' if ATTN_FEED_MODEL_PREDICTIONS else 'true parities'}")
print(f"Recomputed weights reproduce each block's attention output to "
      f"{attn_verify_recomputation():.2e}")

In [22]:
ATTN_SAMPLE_INDEX = 0  # Row of the saved test set to inspect.

attn_x_one = attn_test_data.x[ATTN_SAMPLE_INDEX : ATTN_SAMPLE_INDEX + 1]
attn_y_one = attn_test_data.y[ATTN_SAMPLE_INDEX : ATTN_SAMPLE_INDEX + 1]

with torch.no_grad():
    attn_pred_one = attn_model(attn_x_one)[0]
attn_single_weights = attention_weights(attn_x_one, attn_y_one)[:, 0].float().cpu().numpy()

print(f"input bits: {[int(v) for v in attn_x_one[0].tolist()]}")
print(f"{'target':>8} {'true':>7} {'predicted':>11}")
for name, true_value, predicted in zip(attn_target_names, attn_y_one[0].tolist(), attn_pred_one.tolist()):
    print(f"{name:>8} {true_value:>7.0f} {predicted:>11.4f}")

plot_attention_grid(
    attn_single_weights,
    f"Attention, single test input (row {ATTN_SAMPLE_INDEX}), N={ATTN_N}, "
    f"checkpoint={attn_checkpoint_label}",
    f"attention_single_N_{ATTN_N}_{attn_checkpoint_label}_row_{ATTN_SAMPLE_INDEX}.png",
)

In [23]:
ATTN_AVERAGE_NUM_SAMPLES = 2_048  # Random draw from the saved test set.
ATTN_AVERAGE_SEED = 0

if ATTN_AVERAGE_NUM_SAMPLES > attn_test_data.x.shape[0]:
    raise ValueError(
        f"Requested {ATTN_AVERAGE_NUM_SAMPLES} samples but the test set has "
        f"{attn_test_data.x.shape[0]}"
    )

attn_generator = torch.Generator(device="cpu").manual_seed(ATTN_AVERAGE_SEED)
attn_subset = torch.randperm(attn_test_data.x.shape[0], generator=attn_generator)[
    :ATTN_AVERAGE_NUM_SAMPLES
].to(attn_device)

attn_accumulator = torch.zeros(
    (attn_num_layers, attn_num_heads, attn_num_positions, attn_num_positions),
    device=attn_device,
    dtype=torch.float64,
)
attn_seen = 0
with torch.no_grad():
    for start in range(0, attn_subset.numel(), attn_batch_size):
        idx = attn_subset[start : start + attn_batch_size]
        weights = attention_weights(attn_test_data.x[idx], attn_test_data.y[idx])
        attn_accumulator += weights.to(dtype=torch.float64).sum(dim=1)
        attn_seen += idx.numel()

attn_mean_weights = (attn_accumulator / attn_seen).float().cpu().numpy()

# Every query row is a distribution over its allowed keys, so each row must still
# sum to 1 after averaging.
attn_row_sums = attn_mean_weights.sum(axis=-1)
print(f"Averaged over {attn_seen} samples; row sums in "
      f"[{attn_row_sums.min():.6f}, {attn_row_sums.max():.6f}] (expect 1)")

plot_attention_grid(
    attn_mean_weights,
    f"Attention averaged over {attn_seen} test inputs, N={ATTN_N}, "
    f"checkpoint={attn_checkpoint_label}",
    f"attention_mean_{attn_seen}_N_{ATTN_N}_{attn_checkpoint_label}.png",
)

In [24]:
import pandas as pd

ATTN_TABLE_SOURCE = "mean"  # "mean" uses the averaged weights, "single" the single input.
ATTN_TABLE_DECIMALS = 4
ATTN_TABLE_SAVE = True


def attention_dataframe(weights, layer_idx, head_idx):
    """One (query, key) attention matrix as a labelled DataFrame.

    Rows follow ATTN_QUERY_ROWS, so by default only the target-emitting positions.
    Causally masked cells are NaN rather than 0.0 - the model cannot attend there.
    """
    mask = attn_causal_mask.detach().cpu().numpy()
    rows = attn_row_slice()
    values = np.where(mask, weights[layer_idx, head_idx], np.nan)[rows]
    return pd.DataFrame(values, index=attn_query_labels[rows], columns=attn_key_labels)


if ATTN_TABLE_SOURCE == "mean":
    if "attn_mean_weights" not in globals():
        raise NameError("Run the averaged-attention cell first")
    attn_table_weights = attn_mean_weights
    attn_table_description = f"mean over {attn_seen} test inputs"
    attn_table_tag = f"mean_{attn_seen}"
elif ATTN_TABLE_SOURCE == "single":
    if "attn_single_weights" not in globals():
        raise NameError("Run the single-input attention cell first")
    attn_table_weights = attn_single_weights
    attn_table_description = f"test row {ATTN_SAMPLE_INDEX}"
    attn_table_tag = f"row_{ATTN_SAMPLE_INDEX}"
else:
    raise ValueError('ATTN_TABLE_SOURCE must be "mean" or "single"')

attention_tables = {}
for layer_idx in range(attn_num_layers):
    for head_idx in range(attn_num_heads):
        frame = attention_dataframe(attn_table_weights, layer_idx, head_idx)
        attention_tables[(layer_idx, head_idx)] = frame

        head_suffix = f", head {head_idx + 1}" if attn_num_heads > 1 else ""
        print(f"\nBlock {layer_idx + 1}{head_suffix} - attention weights, {attn_table_description}")
        print(f"rows sum to {frame.sum(axis=1).min():.6f}..{frame.sum(axis=1).max():.6f}")
        with pd.option_context("display.max_columns", None, "display.width", 250):
            display(frame.round(ATTN_TABLE_DECIMALS))

        if ATTN_TABLE_SAVE:
            path = (
                ANALYSIS_DIR
                / f"attention_values_{attn_table_tag}_N_{ATTN_N}_{attn_checkpoint_label}"
                f"_block_{layer_idx}_head_{head_idx}.csv"
            )
            frame.to_csv(path)
            print(f"Saved {path}")

print(f"\n{len(attention_tables)} matrices available as attention_tables[(block_idx, head_idx)]")

In [25]:
import pandas as pd

ATTN_LOGIT_SOURCE = "single"  # "mean" averages over the subset, "single" uses one input.
ATTN_LOGIT_APPLY_SCALE = False  # True: the value softmax actually sees (q.k * logit_scale).
                               # False: the raw q.k dot product.
ATTN_LOGIT_DECIMALS = 4
ATTN_LOGIT_SAVE = True


@torch.no_grad()
def attention_logits(x_batch, y_batch, apply_scale=True):
    """(layers, batch, heads, S, S) pre-softmax scores.

    The same recomputation attention_weights does, stopping before the causal mask
    and the softmax.
    """
    seq_targets = attn_sequence_targets(x_batch, y_batch)
    _, activations = attn_model(x_batch, targets=seq_targets, return_activations=True)
    per_layer = []
    for layer_idx, block in enumerate(attn_model.blocks):
        attention = block.attention
        h = activations[layer_idx]
        q = attention._split_heads(attention.q_proj(h))
        k = attention._split_heads(attention.k_proj(h))
        scores = q @ k.transpose(-1, -2)
        if apply_scale:
            scores = scores * attention.logit_scale
        per_layer.append(scores)
    return torch.stack(per_layer)


@torch.no_grad()
def attn_verify_logits(num_samples=8):
    """Masking and softmaxing these scores must reproduce attention_weights."""
    x_batch = attn_test_data.x[:num_samples]
    y_batch = attn_test_data.y[:num_samples]
    scaled = attention_logits(x_batch, y_batch, apply_scale=True)
    rebuilt = scaled.masked_fill(~attn_causal_mask, float("-inf")).softmax(dim=-1)
    return float((rebuilt - attention_weights(x_batch, y_batch)).abs().max())


def logit_dataframe(scores, layer_idx, head_idx):
    """One (query, key) score matrix as a labelled DataFrame. Masked positions are NaN:
    the model never scores them, it forces them to -inf before the softmax."""
    mask = attn_causal_mask.detach().cpu().numpy()
    rows = attn_row_slice()
    values = np.where(mask, scores[layer_idx, head_idx], np.nan)[rows]
    return pd.DataFrame(values, index=attn_query_labels[rows], columns=attn_key_labels)


if ATTN_LOGIT_SOURCE == "single":
    attn_logit_x = attn_test_data.x[ATTN_SAMPLE_INDEX : ATTN_SAMPLE_INDEX + 1]
    attn_logit_y = attn_test_data.y[ATTN_SAMPLE_INDEX : ATTN_SAMPLE_INDEX + 1]
    attn_logit_scores = (
        attention_logits(attn_logit_x, attn_logit_y, ATTN_LOGIT_APPLY_SCALE)[:, 0]
        .float()
        .cpu()
        .numpy()
    )
    attn_logit_description = f"test row {ATTN_SAMPLE_INDEX}"
    attn_logit_tag = f"row_{ATTN_SAMPLE_INDEX}"
elif ATTN_LOGIT_SOURCE == "mean":
    if "attn_subset" not in globals():
        raise NameError("Run the averaged-attention cell first (it defines the subset)")
    accumulator = torch.zeros(
        (attn_num_layers, attn_num_heads, attn_num_positions, attn_num_positions),
        device=attn_device,
        dtype=torch.float64,
    )
    counted = 0
    with torch.no_grad():
        for start in range(0, attn_subset.numel(), attn_batch_size):
            idx = attn_subset[start : start + attn_batch_size]
            scores = attention_logits(
                attn_test_data.x[idx], attn_test_data.y[idx], ATTN_LOGIT_APPLY_SCALE
            )
            accumulator += scores.to(dtype=torch.float64).sum(dim=1)
            counted += idx.numel()
    # Note: this is the mean of the scores, not the scores of the mean input, and
    # softmax does not commute with averaging - use the weights table for that.
    attn_logit_scores = (accumulator / counted).float().cpu().numpy()
    attn_logit_description = f"mean over {counted} test inputs"
    attn_logit_tag = f"mean_{counted}"
else:
    raise ValueError('ATTN_LOGIT_SOURCE must be "mean" or "single"')

attn_logit_scale_value = attn_model.blocks[0].attention.logit_scale
print(f"attention_logit_scale = {attn_model_config.get('attention_logit_scale')} "
      f"-> multiplier {attn_logit_scale_value:.6g}"
      f"{' (applied)' if ATTN_LOGIT_APPLY_SCALE else ' (NOT applied: raw q.k shown)'}")
print(f"softmax of these scores reproduces the weights to {attn_verify_logits():.2e}")

attention_logit_tables = {}
for layer_idx in range(attn_num_layers):
    for head_idx in range(attn_num_heads):
        frame = logit_dataframe(attn_logit_scores, layer_idx, head_idx)
        attention_logit_tables[(layer_idx, head_idx)] = frame

        head_suffix = f", head {head_idx + 1}" if attn_num_heads > 1 else ""
        flat = frame.to_numpy()
        print(f"\nBlock {layer_idx + 1}{head_suffix} - pre-softmax scores, {attn_logit_description}")
        print(f"range [{np.nanmin(flat):.4g}, {np.nanmax(flat):.4g}], std {np.nanstd(flat):.4g}; "
              f"per-row spread max-min in "
              f"[{np.nanmin(np.nanmax(flat, 1) - np.nanmin(flat, 1)):.4g}, "
              f"{np.nanmax(np.nanmax(flat, 1) - np.nanmin(flat, 1)):.4g}]")
        with pd.option_context("display.max_columns", None, "display.width", 250):
            display(frame.round(ATTN_LOGIT_DECIMALS))

        if ATTN_LOGIT_SAVE:
            path = (
                ANALYSIS_DIR
                / f"attention_logits_{attn_logit_tag}_N_{ATTN_N}_{attn_checkpoint_label}"
                f"_block_{layer_idx}_head_{head_idx}.csv"
            )
            frame.to_csv(path)
            print(f"Saved {path}")

print(f"\n{len(attention_logit_tables)} score matrices available as "
      "attention_logit_tables[(block_idx, head_idx)]")

In [26]:
attn_model_config["attention_logit_scale"] 

## Walsh Modes Inside the MLP

Probe two points inside every block's MLP, both **before** the residual addition:
`after_nonlinearity` is `phi(V x)`, and `after_linear` is `W phi(V x)` when
`use_post_activation_linear` is on (otherwise the two coincide).

At each point, project the vector-valued function `r(x)` onto the target Walsh
monomials `m(x)` and report the norm of the Fourier vector-coefficient
`E_x[r(x) m(x)]`, for every position from the last input bit onward. The sequence is
generated autoregressively -- no teacher forcing -- so the answer positions hold the
model's own predictions.

The cosine tables compare each mode vector against the readout. The readout acts on
the post-linear features, so for `after_nonlinearity` the direction that actually
matters is `W^T w`, because `<w, W a> = <W^T w, a>`. Both are reported; the
"effective" table is the one to read.

In [28]:
import pandas as pd

from parity_net.data import tree_parity_specs

WALSH_NUM_SAMPLES = 8_192  # Random draw from the saved test set.
WALSH_SEED = 0
WALSH_INCLUDE_CONSTANT = True  # Adds the m(x)=1 mode, i.e. the mean of the activation.
WALSH_SAVE = True

if "attn_model" not in globals():
    raise NameError("Run the attention-pattern setup cell first")

walsh_task_config = attn_config.get("task") or {
    "relevant_dim": attn_model.config.relevant_dim,
    "exclude_targets": [],
}
walsh_specs = tree_parity_specs(
    int(walsh_task_config["relevant_dim"]),
    walsh_task_config.get("exclude_targets") or [],
)
walsh_mode_names = [spec.name for spec in walsh_specs]
walsh_mode_indices = [spec.indices for spec in walsh_specs]
if WALSH_INCLUDE_CONSTANT:
    walsh_mode_names = ["const"] + walsh_mode_names
    walsh_mode_indices = [None] + walsh_mode_indices

# Every position from the last input bit onward, i.e. the ones that emit a target.
walsh_positions = list(range(attn_input_dim - 1, attn_num_positions))
walsh_position_labels = [
    attn_target_names[p - (attn_input_dim - 1)] for p in walsh_positions
]
walsh_position_index = torch.tensor(walsh_positions, device=attn_device, dtype=torch.long)

# Two probe points per block, both BEFORE the residual addition.
WALSH_POINTS = ("after_nonlinearity", "after_linear")


def walsh_mode_values(x_batch):
    """(batch, modes) Walsh monomials of the *input bits*."""
    columns = []
    for indices in walsh_mode_indices:
        if indices is None:
            columns.append(torch.ones(x_batch.shape[0], device=x_batch.device, dtype=x_batch.dtype))
        else:
            idx = torch.tensor(indices, device=x_batch.device, dtype=torch.long)
            columns.append(torch.prod(x_batch[:, idx], dim=1))
    return torch.stack(columns, dim=1)


@torch.no_grad()
def block_mlp_internals(x_batch):
    """Replay the autoregressive sequence and capture each block's MLP internals.

    No teacher forcing: the answer positions are filled with the model's own
    generated predictions. Returns a list of (after_nonlinearity, after_linear) per
    block, each (batch, seq, N) and taken before the residual addition, plus the
    final residual stream.
    """
    predictions = attn_model.generate(x_batch)
    fed_back = attn_model.feedback_value(predictions)
    values = (
        x_batch
        if attn_model.output_dim == 1
        else torch.cat([x_batch, fed_back[:, :-1]], dim=1)
    )
    h = attn_model.embed(values)
    captures = []
    for block in attn_model.blocks:
        h = h + block.attention(h)
        after_nonlinearity = block.mlp.activation(block.mlp.linear(h))
        after_linear = (
            block.mlp.post_activation_linear(after_nonlinearity)
            if block.mlp.post_activation_linear is not None
            else after_nonlinearity
        )
        captures.append((after_nonlinearity, after_linear))
        h = h + after_linear
    return captures, h, predictions


@torch.no_grad()
def walsh_verify_replay(num_samples=8):
    """Reading out the replayed stream must reproduce the autoregressive predictions,
    which is what proves the capture points sit on the real forward pass."""
    x_batch = attn_test_data.x[:num_samples]
    _, h, predictions = block_mlp_internals(x_batch)
    out = attn_model.readout(h[:, attn_input_dim - 1 :, :]).squeeze(-1)
    return float((out - predictions).abs().max())


if WALSH_NUM_SAMPLES > attn_test_data.x.shape[0]:
    raise ValueError(
        f"Requested {WALSH_NUM_SAMPLES} samples but the test set has {attn_test_data.x.shape[0]}"
    )
walsh_generator = torch.Generator(device="cpu").manual_seed(WALSH_SEED)
walsh_subset = torch.randperm(attn_test_data.x.shape[0], generator=walsh_generator)[
    :WALSH_NUM_SAMPLES
].to(attn_device)

# E_x[ r(x) m(x) ] for every (block, probe point, position, mode).
walsh_accumulator = torch.zeros(
    (attn_num_layers, len(WALSH_POINTS), len(walsh_positions), len(walsh_mode_names), attn_model.config.N),
    device=attn_device,
    dtype=torch.float64,
)
walsh_seen = 0
with torch.no_grad():
    for start in range(0, walsh_subset.numel(), attn_batch_size):
        idx = walsh_subset[start : start + attn_batch_size]
        x_batch = attn_test_data.x[idx]
        modes = walsh_mode_values(x_batch).to(dtype=torch.float64)
        captures, _, _ = block_mlp_internals(x_batch)
        for layer_idx, per_point in enumerate(captures):
            for point_idx, tensor in enumerate(per_point):
                selected = tensor[:, walsh_position_index, :].to(dtype=torch.float64)
                walsh_accumulator[layer_idx, point_idx] += torch.einsum(
                    "bm,bpn->pmn", modes, selected
                )
        walsh_seen += idx.numel()

walsh_vectors = (walsh_accumulator / walsh_seen).cpu()
walsh_norms = walsh_vectors.norm(dim=-1).numpy()

# Readout direction, and the direction that actually acts on the pre-linear features:
# <w, W a> = <W^T w, a>, so W^T w is the effective functional after the nonlinearity.
walsh_readout = attn_model.readout.weight[0].detach().to(dtype=torch.float64).cpu()
walsh_effective = []
for block in attn_model.blocks:
    post_linear = block.mlp.post_activation_linear
    walsh_effective.append(
        post_linear.weight.detach().to(dtype=torch.float64).cpu().T @ walsh_readout
        if post_linear is not None
        else walsh_readout.clone()
    )


def walsh_cosine(vectors, direction):
    numerator = vectors @ direction
    denominator = vectors.norm(dim=-1) * direction.norm()
    return (numerator / denominator.clamp_min(1e-30)).numpy()


walsh_cos_readout = np.stack(
    [
        np.stack([walsh_cosine(walsh_vectors[l, p], walsh_readout) for p in range(len(WALSH_POINTS))])
        for l in range(attn_num_layers)
    ]
)
walsh_cos_effective = np.stack(
    [
        np.stack(
            [
                walsh_cosine(walsh_vectors[l, 0], walsh_effective[l]),
                walsh_cosine(walsh_vectors[l, 1], walsh_readout),
            ]
        )
        for l in range(attn_num_layers)
    ]
)

print(f"Samples: {walsh_seen} (autoregressive, no teacher forcing)")
print(f"Replayed stream reproduces the generated predictions to {walsh_verify_replay():.2e}")
print(f"Modes: {walsh_mode_names}")
print(f"Positions: {walsh_position_labels}")
if attn_model.blocks[0].mlp.post_activation_linear is None:
    print("NOTE: use_post_activation_linear is False, so the two probe points coincide.")
print("NOTE: cos vs readout is exact only for the last block; earlier blocks still pass")
print("      through later blocks before the readout.")

for layer_idx in range(attn_num_layers):
    for point_idx, point in enumerate(WALSH_POINTS):
        frame = pd.DataFrame(
            walsh_norms[layer_idx, point_idx],
            index=walsh_position_labels,
            columns=walsh_mode_names,
        )
        frame.index.name = "position emits"
        print(f"\nBlock {layer_idx + 1}, {point} - ||E_x[r(x) m(x)]||")
        with pd.option_context("display.max_columns", None, "display.width", 220):
            display(frame.round(4))
        if WALSH_SAVE:
            path = (
                ANALYSIS_DIR
                / f"walsh_mode_norms_N_{ATTN_N}_{attn_checkpoint_label}"
                f"_block_{layer_idx}_{point}.csv"
            )
            frame.to_csv(path)
            print(f"Saved {path}")

In [34]:
WALSH_POSITION_TARGET = "d8_0"  # Which prediction position to inspect.

if WALSH_POSITION_TARGET not in walsh_position_labels:
    raise ValueError(
        f"{WALSH_POSITION_TARGET} is not an emitted target; choose from {walsh_position_labels}"
    )
walsh_row = walsh_position_labels.index(WALSH_POSITION_TARGET)

print(f"Position that emits {WALSH_POSITION_TARGET} "
      f"(sequence index {walsh_positions[walsh_row]})\n")

rows, norm_rows, cos_readout_rows, cos_effective_rows = [], [], [], []
for layer_idx in range(attn_num_layers):
    for point_idx, point in enumerate(WALSH_POINTS):
        rows.append(f"block {layer_idx + 1} / {point}")
        norm_rows.append(walsh_norms[layer_idx, point_idx, walsh_row])
        cos_readout_rows.append(walsh_cos_readout[layer_idx, point_idx, walsh_row])
        cos_effective_rows.append(walsh_cos_effective[layer_idx, point_idx, walsh_row])

def walsh_table(values, title, filename):
    frame = pd.DataFrame(values, index=rows, columns=walsh_mode_names)
    print(title)
    with pd.option_context("display.max_columns", None, "display.width", 220):
        display(frame.round(4))
    if WALSH_SAVE:
        path = ANALYSIS_DIR / filename
        frame.to_csv(path)
        print(f"Saved {path}\n")
    return frame

walsh_norm_table = walsh_table(
    norm_rows,
    "Mode norms  ||E_x[r(x) m(x)]||",
    f"walsh_norms_{WALSH_POSITION_TARGET}_N_{ATTN_N}_{attn_checkpoint_label}.csv",
)
walsh_cos_readout_table = walsh_table(
    cos_readout_rows,
    "Cosine with the readout direction w",
    f"walsh_cos_readout_{WALSH_POSITION_TARGET}_N_{ATTN_N}_{attn_checkpoint_label}.csv",
)
walsh_cos_effective_table = walsh_table(
    cos_effective_rows,
    "Cosine with the EFFECTIVE direction: W^T w after the nonlinearity, w after the linear.\n"
    "This is the one to read - <w, W a> = <W^T w, a>, so W^T w is what the readout\n"
    "actually measures on the post-nonlinearity features.",
    f"walsh_cos_effective_{WALSH_POSITION_TARGET}_N_{ATTN_N}_{attn_checkpoint_label}.csv",
)

# Signed scalar this block's update contributes to the output along w.
print("Signed contribution to the output:  <w, E_x[r(x) m(x)]>  (after_linear rows only)")
contribution = pd.DataFrame(
    [
        (walsh_vectors[l, 1, walsh_row] @ walsh_readout).numpy()
        for l in range(attn_num_layers)
    ],
    index=[f"block {l + 1}" for l in range(attn_num_layers)],
    columns=walsh_mode_names,
)
with pd.option_context("display.max_columns", None, "display.width", 220):
    display(contribution.round(4))
print(f"target mode here is {WALSH_POSITION_TARGET}; a correct readout wants that column "
      "near 1 and the others near 0")

## Comparing Target Modes Across Positions

`walsh-compute` gives a mode vector `E_x[r(x) m(x)]` for every (block, probe point,
position, mode). This block reads that same tensor across *positions*: for each target
mode it reports the norm at every emitting position, and the position-by-position
cosine similarity matrix of that one mode's vector.

Two questions this answers. Does a mode such as `d2_0` live only at the position that
emits it, or is it carried along the whole sequence? And when it appears at several
positions, is it the *same* direction in the residual stream or a different one?

Norms use a sequential scale from zero; cosines use a diverging scale centred on zero,
since their sign is meaningful.

In [ ]:
if "walsh_vectors" not in globals():
    raise NameError("Run the Walsh-mode cell first")

WALSH_CROSS_MODE = "d2_0"  # Mode whose cosine table is printed in full.
WALSH_CROSS_BLOCK = 0  # Block used for the per-mode cosine figure.
WALSH_CROSS_POINT = "after_linear"  # or "after_nonlinearity"
WALSH_CROSS_SAVE = True

if WALSH_CROSS_MODE not in walsh_mode_names:
    raise ValueError(f"{WALSH_CROSS_MODE} is not one of {walsh_mode_names}")
if WALSH_CROSS_POINT not in WALSH_POINTS:
    raise ValueError(f"WALSH_CROSS_POINT must be one of {WALSH_POINTS}")
if not 0 <= WALSH_CROSS_BLOCK < attn_num_layers:
    raise ValueError(f"WALSH_CROSS_BLOCK must be in [0, {attn_num_layers - 1}]")

# Cosine between the SAME mode's vector at two different positions.
# walsh_vectors is (blocks, points, positions, modes, N).
walsh_unit = walsh_vectors / walsh_vectors.norm(dim=-1, keepdim=True).clamp_min(1e-30)
walsh_position_cosines = torch.einsum(
    "lqpmn,lqrmn->lqmpr", walsh_unit, walsh_unit
).numpy()

print(f"Mode vectors: {tuple(walsh_vectors.shape)} (blocks, points, positions, modes, N)")
print(f"Cross-position cosines: {walsh_position_cosines.shape} "
      "(blocks, points, modes, position, position)")
print(f"Positions: {walsh_position_labels}")
print(f"Modes: {walsh_mode_names}")

mode_idx = walsh_mode_names.index(WALSH_CROSS_MODE)
point_idx = WALSH_POINTS.index(WALSH_CROSS_POINT)

norms_by_position = pd.DataFrame(
    walsh_norms[WALSH_CROSS_BLOCK, point_idx],
    index=walsh_position_labels,
    columns=walsh_mode_names,
)
norms_by_position.index.name = "position emits"
print(f"\nBlock {WALSH_CROSS_BLOCK + 1}, {WALSH_CROSS_POINT} - mode norms by position")
with pd.option_context("display.max_columns", None, "display.width", 220):
    display(norms_by_position.round(4))

cross_frame = pd.DataFrame(
    walsh_position_cosines[WALSH_CROSS_BLOCK, point_idx, mode_idx],
    index=walsh_position_labels,
    columns=walsh_position_labels,
)
print(f"\nBlock {WALSH_CROSS_BLOCK + 1}, {WALSH_CROSS_POINT} - cosine of the "
      f"{WALSH_CROSS_MODE} mode vector between positions")
with pd.option_context("display.max_columns", None, "display.width", 220):
    display(cross_frame.round(4))

if WALSH_CROSS_SAVE:
    prefix = (
        f"walsh_cross_N_{ATTN_N}_{attn_checkpoint_label}"
        f"_block_{WALSH_CROSS_BLOCK}_{WALSH_CROSS_POINT}"
    )
    norms_by_position.to_csv(ANALYSIS_DIR / f"{prefix}_norms.csv")
    cross_frame.to_csv(ANALYSIS_DIR / f"{prefix}_cosines_{WALSH_CROSS_MODE}.csv")
    print(f"Saved {ANALYSIS_DIR / prefix}_norms.csv and _cosines_{WALSH_CROSS_MODE}.csv")

In [ ]:
WALSH_CROSS_ANNOTATE = True

# ---- 1. mode norms: positions x modes, one panel per (block, probe point) ----
norms_max = float(np.nanmax(walsh_norms)) or 1.0
norm_cmap = plt.get_cmap("Blues")
cell = 0.62
fig, axes = plt.subplots(
    len(WALSH_POINTS),
    attn_num_layers,
    figsize=(
        (cell * len(walsh_mode_names) + 2.2) * attn_num_layers,
        (cell * len(walsh_position_labels) + 2.0) * len(WALSH_POINTS),
    ),
    squeeze=False,
    constrained_layout=True,
)
for layer_idx in range(attn_num_layers):
    for point_i, point in enumerate(WALSH_POINTS):
        ax = axes[point_i][layer_idx]
        values = walsh_norms[layer_idx, point_i]
        image = ax.imshow(values, cmap=norm_cmap, vmin=0.0, vmax=norms_max, interpolation="nearest")
        ax.set_title(f"block {layer_idx + 1}, {point}", fontsize=10)
        ax.set_xticks(range(len(walsh_mode_names)))
        ax.set_xticklabels(walsh_mode_names, rotation=90, fontsize=8)
        ax.set_yticks(range(len(walsh_position_labels)))
        ax.set_yticklabels(walsh_position_labels, fontsize=8)
        if layer_idx == 0:
            ax.set_ylabel("position emits", fontsize=9)
        ax.set_xlabel("mode", fontsize=9)
        ax.tick_params(length=0)
        if WALSH_CROSS_ANNOTATE:
            for r in range(values.shape[0]):
                for c in range(values.shape[1]):
                    ax.text(c, r, f"{values[r, c]:.2f}", ha="center", va="center", fontsize=6,
                            color="white" if values[r, c] > 0.6 * norms_max else "black")
fig.colorbar(image, ax=axes, label="||E_x[r(x) m(x)]||", fraction=0.02, pad=0.01)
fig.suptitle(f"Mode norms by position, N={ATTN_N}, checkpoint={attn_checkpoint_label}",
             fontsize=12)
if WALSH_CROSS_SAVE:
    path = ANALYSIS_DIR / f"walsh_cross_norms_N_{ATTN_N}_{attn_checkpoint_label}.png"
    fig.savefig(path, dpi=170, bbox_inches="tight")
    print(f"Saved {path}")
plt.show()

# ---- 2. per-mode cosine between positions, for the selected block and point ----
point_idx = WALSH_POINTS.index(WALSH_CROSS_POINT)
columns = min(4, len(walsh_mode_names))
rows = (len(walsh_mode_names) + columns - 1) // columns
panel = 0.55 * len(walsh_position_labels) + 2.0
fig, axes = plt.subplots(rows, columns, figsize=(panel * columns, panel * rows),
                         squeeze=False, constrained_layout=True)
for slot in range(rows * columns):
    ax = axes[slot // columns][slot % columns]
    if slot >= len(walsh_mode_names):
        ax.axis("off")
        continue
    values = walsh_position_cosines[WALSH_CROSS_BLOCK, point_idx, slot]
    image = ax.imshow(values, cmap="coolwarm", vmin=-1.0, vmax=1.0, interpolation="nearest")
    ax.set_title(walsh_mode_names[slot], fontsize=10)
    ax.set_xticks(range(len(walsh_position_labels)))
    ax.set_xticklabels(walsh_position_labels, rotation=90, fontsize=7)
    ax.set_yticks(range(len(walsh_position_labels)))
    ax.set_yticklabels(walsh_position_labels, fontsize=7)
    ax.tick_params(length=0)
    if WALSH_CROSS_ANNOTATE:
        for r in range(values.shape[0]):
            for c in range(values.shape[1]):
                ax.text(c, r, f"{values[r, c]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if abs(values[r, c]) > 0.55 else "black")
fig.colorbar(image, ax=axes, label="cosine between positions", fraction=0.02, pad=0.01)
fig.suptitle(
    f"Same mode, different positions - block {WALSH_CROSS_BLOCK + 1}, "
    f"{WALSH_CROSS_POINT}, N={ATTN_N}, checkpoint={attn_checkpoint_label}",
    fontsize=12,
)
if WALSH_CROSS_SAVE:
    path = (
        ANALYSIS_DIR
        / f"walsh_cross_cosines_N_{ATTN_N}_{attn_checkpoint_label}"
        f"_block_{WALSH_CROSS_BLOCK}_{WALSH_CROSS_POINT}.png"
    )
    fig.savefig(path, dpi=170, bbox_inches="tight")
    print(f"Saved {path}")
plt.show()

print("Diagonal is 1 by construction. Off-diagonal near 1 means the mode occupies the "
      "same residual-stream direction at both positions; near 0 means different "
      "directions; the norm heatmap says whether it is present at all.")

## Decoding the Computation of a 4-Degree Parity

The transformer analogue of the partition decoding in
`colab_width_scaling_sgd_exclude_d8_d16.ipynb`.

Pick the position that emits a degree-4 target and a block. Take the residual stream
entering that block's MLP at that position -- after the block's attention, before
`linear` -- and Fourier-decompose it over the 15 non-empty subsets of the four parity
indices. For each set partition of those indices, rebuild the MLP input from only the
modes named by that partition, push it through the MLP, and ask how much of the
original degree-4 output mode survives.

Intervening after attention is deliberate. If the stream were replaced before
attention, the other positions would re-inject the modes just removed and the
decomposition would be meaningless.

The sequence is generated autoregressively -- no teacher forcing -- and cached once,
so every partition is evaluated on exactly the same inputs.

Two rankings are reported per partition: the cosine of the reconstructed degree-4
output mode with the original one, and the MSE of the model's actual prediction at
that position under the intervention.

In [40]:
import itertools

import pandas as pd
import torch.nn.functional as F

DECODE4_TARGET = "d4_0"  # Which degree-4 target's position to decode.
DECODE4_BLOCK_IDX = 0  # Block whose MLP input is decomposed.
DECODE4_NUM_SAMPLES = 8_192 * 2
DECODE4_SEED = 0
DECODE4_INCLUDE_CONSTANT = True  # Keep the mean of the MLP input in the reconstruction.
DECODE4_EPS = 1e-12
DECODE4_SAVE = True

if "attn_model" not in globals():
    raise NameError("Run the attention-pattern setup cell first")
if DECODE4_TARGET not in attn_target_names:
    raise ValueError(f"{DECODE4_TARGET} is not an active target; pick from {attn_target_names}")
if not 0 <= DECODE4_BLOCK_IDX < attn_num_layers:
    raise ValueError(f"DECODE4_BLOCK_IDX must be in [0, {attn_num_layers - 1}]")

decode4_spec = next(s for s in walsh_specs if s.name == DECODE4_TARGET)
if decode4_spec.degree != 4:
    raise ValueError(f"{DECODE4_TARGET} has degree {decode4_spec.degree}, expected 4")
decode4_indices = tuple(sorted(decode4_spec.indices))
decode4_position = attn_input_dim - 1 + attn_target_names.index(DECODE4_TARGET)
decode4_block = attn_model.blocks[DECODE4_BLOCK_IDX]


def set_partitions(items):
    items = tuple(items)
    if not items:
        yield ()
        return
    first, rest = items[0], items[1:]
    for partition in set_partitions(rest):
        yield ((first,),) + partition
        for block_idx in range(len(partition)):
            merged = tuple(sorted(partition[block_idx] + (first,)))
            yield partition[:block_idx] + (merged,) + partition[block_idx + 1 :]


def canonical_partition(partition):
    return tuple(
        sorted((tuple(sorted(block)) for block in partition), key=lambda b: (len(b), b))
    )


def partition_label(partition):
    return " | ".join("*".join(f"x{i + 1}" for i in block) for block in partition)


def mode_label(mode):
    return "*".join(f"x{i + 1}" for i in mode)


decode4_partitions = sorted(
    {canonical_partition(p) for p in set_partitions(decode4_indices)},
    key=lambda p: (len(p), p),
)
decode4_subsets = [
    tuple(combo)
    for size in range(1, len(decode4_indices) + 1)
    for combo in itertools.combinations(decode4_indices, size)
]


def decode4_monomial(x_batch, indices):
    idx = torch.tensor(indices, device=x_batch.device, dtype=torch.long)
    return torch.prod(x_batch[:, idx], dim=1)


@torch.no_grad()
def decode4_forward(values_batch, replacement=None):
    """Run the cached sequence forward, optionally overwriting the MLP input at the
    decoded position in block DECODE4_BLOCK_IDX.

    Returns (mlp_input_at_position, block_update_at_position, prediction_at_position).
    """
    h = attn_model.embed(values_batch)
    mlp_input = None
    block_update = None
    for layer_idx, block in enumerate(attn_model.blocks):
        h = h + block.attention(h)
        if layer_idx == DECODE4_BLOCK_IDX:
            if replacement is not None:
                h = h.clone()
                h[:, decode4_position, :] = replacement
            mlp_input = h[:, decode4_position, :]
        update = block.mlp.activation(block.mlp.linear(h))
        if block.mlp.post_activation_linear is not None:
            update = block.mlp.post_activation_linear(update)
        if layer_idx == DECODE4_BLOCK_IDX:
            block_update = update[:, decode4_position, :]
        h = h + update
    prediction = attn_model.readout(h[:, decode4_position, :]).squeeze(-1)
    return mlp_input, block_update, prediction


# Cache the autoregressive sequence once so every partition sees identical inputs.
if DECODE4_NUM_SAMPLES > attn_test_data.x.shape[0]:
    raise ValueError(
        f"Requested {DECODE4_NUM_SAMPLES} samples but the test set has {attn_test_data.x.shape[0]}"
    )
decode4_generator = torch.Generator(device="cpu").manual_seed(DECODE4_SEED)
decode4_subset = torch.randperm(attn_test_data.x.shape[0], generator=decode4_generator)[
    :DECODE4_NUM_SAMPLES
].to(attn_device)
decode4_x = attn_test_data.x[decode4_subset]

with torch.no_grad():
    chunks = []
    for start in range(0, decode4_x.shape[0], attn_batch_size):
        x_chunk = decode4_x[start : start + attn_batch_size]
        predictions = attn_model.generate(x_chunk)
        fed_back = attn_model.feedback_value(predictions)
        chunks.append(
            x_chunk
            if attn_model.output_dim == 1
            else torch.cat([x_chunk, fed_back[:, :-1]], dim=1)
        )
    decode4_values = torch.cat(chunks, dim=0)

width = attn_model.config.N
decode4_constant = torch.zeros(width, device=attn_device, dtype=torch.float64)
decode4_directions = {s: torch.zeros(width, device=attn_device, dtype=torch.float64) for s in decode4_subsets}
decode4_full_mode = torch.zeros(width, device=attn_device, dtype=torch.float64)
decode4_baseline_sq = 0.0
decode4_seen = 0

with torch.no_grad():
    for start in range(0, decode4_x.shape[0], attn_batch_size):
        stop = min(start + attn_batch_size, decode4_x.shape[0])
        x_chunk = decode4_x[start:stop]
        mlp_input, block_update, prediction = decode4_forward(decode4_values[start:stop])
        target = decode4_monomial(x_chunk, decode4_indices)
        decode4_constant += mlp_input.to(dtype=torch.float64).sum(dim=0)
        for subset in decode4_subsets:
            weights = decode4_monomial(x_chunk, subset).to(dtype=torch.float64)
            decode4_directions[subset] += (weights[:, None] * mlp_input.to(dtype=torch.float64)).sum(dim=0)
        decode4_full_mode += (
            target.to(dtype=torch.float64)[:, None] * block_update.to(dtype=torch.float64)
        ).sum(dim=0)
        decode4_baseline_sq += float(((prediction - target) ** 2).sum())
        decode4_seen += x_chunk.shape[0]

decode4_constant /= decode4_seen
decode4_directions = {s: v / decode4_seen for s, v in decode4_directions.items()}
decode4_full_mode = (decode4_full_mode / decode4_seen).cpu()
decode4_baseline_mse = decode4_baseline_sq / decode4_seen

print(f"Target {DECODE4_TARGET} = {mode_label(decode4_indices)}, "
      f"emitted at sequence position {decode4_position}")
print(f"Block {DECODE4_BLOCK_IDX + 1} of {attn_num_layers}; {decode4_seen} samples, "
      "autoregressive (no teacher forcing)")
print(f"Original degree-4 output mode norm: {decode4_full_mode.norm():.6g}")
print(f"No-intervention MSE at this position: {decode4_baseline_mse:.6g}")
print(f"{len(decode4_partitions)} partitions, {len(decode4_subsets)} candidate input modes")
print("Input-direction norms:")
display(
    pd.DataFrame(
        [
            {"mode": mode_label(s), "norm": float(decode4_directions[s].norm())}
            for s in decode4_subsets
        ]
    ).set_index("mode").round(6)
)

rows = []
with torch.no_grad():
    for partition_idx, partition in enumerate(decode4_partitions):
        active = {
            s: decode4_directions[s].to(dtype=attn_dtype)
            for s in partition
            if float(decode4_directions[s].norm()) > DECODE4_EPS
        }
        constant = decode4_constant.to(dtype=attn_dtype)
        reduced_mode = torch.zeros(width, device=attn_device, dtype=torch.float64)
        squared_error = 0.0
        for start in range(0, decode4_x.shape[0], attn_batch_size):
            stop = min(start + attn_batch_size, decode4_x.shape[0])
            x_chunk = decode4_x[start:stop]
            replacement = torch.zeros((x_chunk.shape[0], width), device=attn_device, dtype=attn_dtype)
            if DECODE4_INCLUDE_CONSTANT:
                replacement = replacement + constant.unsqueeze(0)
            for subset, direction in active.items():
                weights = decode4_monomial(x_chunk, subset).to(dtype=attn_dtype)
                replacement = replacement + weights[:, None] * direction.unsqueeze(0)
            _, block_update, prediction = decode4_forward(decode4_values[start:stop], replacement)
            target = decode4_monomial(x_chunk, decode4_indices)
            reduced_mode += (
                target.to(dtype=torch.float64)[:, None] * block_update.to(dtype=torch.float64)
            ).sum(dim=0)
            squared_error += float(((prediction - target) ** 2).sum())
        reduced_mode = (reduced_mode / decode4_seen).cpu()
        rows.append(
            {
                "partition_idx": partition_idx,
                "partition": partition_label(partition),
                "num_blocks": len(partition),
                "cosine_with_original_mode": float(
                    F.cosine_similarity(
                        reduced_mode.unsqueeze(0), decode4_full_mode.unsqueeze(0), dim=1, eps=DECODE4_EPS
                    )
                ),
                "reduced_mode_norm": float(reduced_mode.norm()),
                "mse": squared_error / decode4_seen,
            }
        )

decode4_df = pd.DataFrame(rows)
decode4_df["cosine_times_norm"] = (
    decode4_df["cosine_with_original_mode"] * decode4_df["reduced_mode_norm"]
)
decode4_df["no_intervention_mse"] = decode4_baseline_mse
decode4_df = decode4_df.sort_values("mse").reset_index(drop=True)

print("\nPartitions ranked by MSE of the actual prediction (lower = this decomposition "
      "carries the computation):")
with pd.option_context("display.max_columns", None, "display.width", 220):
    display(decode4_df.round(6))

if DECODE4_SAVE:
    decode4_path = (
        ANALYSIS_DIR
        / f"decode4_partitions_{DECODE4_TARGET}_N_{ATTN_N}_{attn_checkpoint_label}"
        f"_block_{DECODE4_BLOCK_IDX}.csv"
    )
    decode4_df.to_csv(decode4_path, index=False)
    print(f"Saved {decode4_path}")

In [41]:
import matplotlib.pyplot as plt

DECODE4_PLOT_LOG_MSE = True

ordered = decode4_df.sort_values("mse").reset_index(drop=True)
fig, axes = plt.subplots(2, 1, figsize=(max(9, 0.55 * len(ordered)), 9), sharex=True,
                         constrained_layout=True)

axes[0].bar(range(len(ordered)), ordered["mse"], color="#4C78A8", width=0.7)
axes[0].axhline(decode4_baseline_mse, color="#CC6677", linestyle="--", linewidth=1.5,
                label=f"no intervention = {decode4_baseline_mse:.4g}")
axes[0].set_ylabel(f"MSE for {DECODE4_TARGET}")
if DECODE4_PLOT_LOG_MSE and (ordered["mse"] > 0).all() and decode4_baseline_mse > 0:
    axes[0].set_yscale("log")
axes[0].legend()
axes[0].grid(True, axis="y", alpha=0.25, which="both")
axes[0].set_title(
    f"Partition decoding of {DECODE4_TARGET}, block {DECODE4_BLOCK_IDX + 1}, "
    f"N={ATTN_N}, checkpoint={attn_checkpoint_label}"
)

axes[1].bar(range(len(ordered)), ordered["cosine_with_original_mode"], color="#72B7B2", width=0.7)
axes[1].axhline(0.0, color="#888888", linewidth=0.8)
axes[1].set_ylabel("cosine with original mode")
axes[1].set_ylim(-1.05, 1.05)
axes[1].grid(True, axis="y", alpha=0.25)
axes[1].set_xticks(range(len(ordered)))
axes[1].set_xticklabels(ordered["partition"], rotation=90)
axes[1].set_xlabel("partition of the four indices (sorted by MSE)")

if DECODE4_SAVE:
    plot_path = (
        ANALYSIS_DIR
        / f"decode4_partitions_{DECODE4_TARGET}_N_{ATTN_N}_{attn_checkpoint_label}"
        f"_block_{DECODE4_BLOCK_IDX}.png"
    )
    fig.savefig(plot_path, dpi=170, bbox_inches="tight")
    print(f"Saved {plot_path}")
plt.show()

best = ordered.iloc[0]
print(f"Best partition by MSE:  {best['partition']}")
print(f"  MSE {best['mse']:.6g}  vs  no-intervention {decode4_baseline_mse:.6g}")
print(f"  cosine with the original degree-4 mode: {best['cosine_with_original_mode']:.4f}")
best_cos = ordered.sort_values("cosine_with_original_mode", ascending=False).iloc[0]
print(f"Best partition by cosine: {best_cos['partition']} "
      f"(cos {best_cos['cosine_with_original_mode']:.4f}, mse {best_cos['mse']:.6g})")

In [42]:
DECODE4_CUM_SAVE = True

# Add partitions best-first (by the MSE ranking above), accumulating their modes, and
# watch the prediction recover.
decode4_kept = set()
cumulative_rows = []
constant = decode4_constant.to(dtype=attn_dtype)

with torch.no_grad():
    for step, row in enumerate(decode4_df.itertuples(index=False), start=1):
        partition = decode4_partitions[int(row.partition_idx)]
        new_modes = [m for m in partition if m not in decode4_kept]
        decode4_kept.update(new_modes)
        active = {
            s: decode4_directions[s].to(dtype=attn_dtype)
            for s in sorted(decode4_kept, key=lambda m: (len(m), m))
            if float(decode4_directions[s].norm()) > DECODE4_EPS
        }
        squared_error = 0.0
        for start in range(0, decode4_x.shape[0], attn_batch_size):
            stop = min(start + attn_batch_size, decode4_x.shape[0])
            x_chunk = decode4_x[start:stop]
            replacement = torch.zeros((x_chunk.shape[0], width), device=attn_device, dtype=attn_dtype)
            if DECODE4_INCLUDE_CONSTANT:
                replacement = replacement + constant.unsqueeze(0)
            for subset, direction in active.items():
                weights = decode4_monomial(x_chunk, subset).to(dtype=attn_dtype)
                replacement = replacement + weights[:, None] * direction.unsqueeze(0)
            _, _, prediction = decode4_forward(decode4_values[start:stop], replacement)
            target = decode4_monomial(x_chunk, decode4_indices)
            squared_error += float(((prediction - target) ** 2).sum())
        cumulative_rows.append(
            {
                "step": step,
                "partition": row.partition,
                "new_modes": ", ".join(mode_label(m) for m in new_modes) or "no new modes",
                "num_kept_modes": len(decode4_kept),
                "kept_modes": ", ".join(
                    mode_label(m) for m in sorted(decode4_kept, key=lambda m: (len(m), m))
                ),
                "mse": squared_error / decode4_seen,
                "no_intervention_mse": decode4_baseline_mse,
            }
        )

decode4_cumulative_df = pd.DataFrame(cumulative_rows)
with pd.option_context("display.max_columns", None, "display.width", 240):
    display(decode4_cumulative_df.round(6))

fig, ax = plt.subplots(figsize=(max(9, 0.6 * len(decode4_cumulative_df)), 5))
ax.plot(decode4_cumulative_df["step"], decode4_cumulative_df["mse"], marker="o", linewidth=1.5,
        color="#4C78A8")
ax.axhline(decode4_baseline_mse, color="#CC6677", linestyle="--", linewidth=1.5,
           label=f"no intervention = {decode4_baseline_mse:.4g}")
ax.set_xlabel("cumulative step: newly added modes")
ax.set_ylabel(f"MSE for {DECODE4_TARGET}")
ax.set_xticks(decode4_cumulative_df["step"])
ax.set_xticklabels(decode4_cumulative_df["new_modes"], rotation=90)
if (decode4_cumulative_df["mse"] > 0).all() and decode4_baseline_mse > 0:
    ax.set_yscale("log")
ax.grid(True, alpha=0.25, which="both")
ax.legend()
ax.set_title(f"Cumulative mode reconstruction, {DECODE4_TARGET}, block {DECODE4_BLOCK_IDX + 1}")
fig.tight_layout()

if DECODE4_CUM_SAVE:
    path = (
        ANALYSIS_DIR
        / f"decode4_cumulative_{DECODE4_TARGET}_N_{ATTN_N}_{attn_checkpoint_label}"
        f"_block_{DECODE4_BLOCK_IDX}.csv"
    )
    decode4_cumulative_df.to_csv(path, index=False)
    fig.savefig(path.with_suffix(".png"), dpi=170, bbox_inches="tight")
    print(f"Saved {path}")
plt.show()